In [146]:
import numpy as np
import pandas as pd

In [147]:
from sklearn.impute import SimpleImputer

class DynamicImputer:
    def __init__(self, strategy='mean', columns=None):
        self.strategy = strategy
        self.columns = columns
        self.imputers = {}
    
    def fit(self, data: pd.DataFrame):
        """Fit imputers on training data"""
        cols = self.columns if self.columns else data.columns
        
        for col in cols:
            self.imputers[col] = SimpleImputer(
                missing_values=np.nan, 
                strategy=self.strategy
            )
            self.imputers[col].fit(data[[col]])
        
        return self
    
    def transform(self, data: pd.DataFrame) -> pd.DataFrame:
        """Apply imputation"""
        data = data.copy()
        cols = self.columns if self.columns else data.columns
        
        for col in cols:
            if col in self.imputers:
                data[col] = self.imputers[col].transform(data[[col]])
        
        return data
    
    def fit_transform(self, data: pd.DataFrame) -> pd.DataFrame:
        """Fit and transform in one step"""
        return self.fit(data).transform(data)

In [148]:
import opendatasets as od

dataset_url = 'https://www.kaggle.com/competitions/games-rating'
od.download(dataset_url, force=True)

100%|██████████| 797k/797k [00:00<00:00, 1.33GB/s]


Extracting archive ./games-rating/games-rating.zip to ./games-rating


In [149]:
train = pd.read_csv('./games-rating/train_data.csv', index_col=0)
test = pd.read_csv('./games-rating/test_data.csv', index_col=0)

test

,Name,Year Published,Min Players,Max Players,Play Time,Min Age,Users Rated,BGG Rank,Complexity Average,Owned Users,Mechanics,Domains
ID,,,,,,,,,,,,
161936.0,Pandemic Legacy: Season 1,2015.0,2,4,60,13,41643,2,"2,84",65294.0,"Action Points, Cooperative Game, Hand Manageme...","Strategy Games, Thematic Games"
12333.0,Twilight Struggle,2005.0,2,2,180,13,40814,10,"3,59",56219.0,"Action/Event, Advantage Token, Area Majority /...","Strategy Games, Wargames"
115746.0,War of the Ring: Second Edition,2012.0,2,4,180,13,13725,12,"4,14",22281.0,"Area Majority / Influence, Area Movement, Camp...","Thematic Games, Wargames"
169786.0,Scythe,2016.0,1,5,115,14,57871,14,"3,41",75640.0,"Area Majority / Influence, Card Play Conflict ...",Strategy Games
28720.0,Brass: Lancashire,2007.0,2,4,120,14,19400,19,"3,86",25429.0,"Hand Management, Income, Loans, Network and Ro...",Strategy Games
...,...,...,...,...,...,...,...,...,...,...,...,...
6932.0,Hi Ho! Cherry-O,1960.0,2,4,10,3,1035,20325,"1,03",1691.0,"Cooperative Game, Roll / Spin and Move",Children's Games
3510.0,Battle of the Sexes,1997.0,2,8,45,12,1090,20328,"1,08",1987.0,Team-Based Game,Party Games
5895.0,Hungry Hungry Hippos,1978.0,2,4,10,4,2361,20330,"1,05",2568.0,NaN,Children's Games


In [150]:
def convertion(data: pd.DataFrame, fit: bool=False) -> pd.DataFrame:
    data = data.drop(columns=["Name"])

    data["Complexity Average"] = data["Complexity Average"].str.replace(',', '.').astype(float)
    if fit:
        data["Rating Average"] = data["Rating Average"].str.replace(',', '.').astype(float)

    return data


imputer = DynamicImputer(strategy='mean')

def imputing(data: pd.DataFrame, fit: bool=False) -> pd.DataFrame:
    if fit:
        data = imputer.fit_transform(data)
    else:
        data = imputer.transform(data)

    return data

from sklearn.preprocessing import MultiLabelBinarizer
mlb_mech = MultiLabelBinarizer()
mlb_dom = MultiLabelBinarizer()

def expanding(data: pd.DataFrame, fit: bool=False) -> pd.DataFrame:
    data = data.copy()
    data['Mechanics'] = data['Mechanics'].fillna('').apply(lambda x: [m.strip() for m in x.split(',') if m])
    data['Domains'] = data['Domains'].fillna('').apply(lambda x: [d.strip() for d in x.split(',') if d])

    if fit:
        mech_encoded = mlb_mech.fit_transform(data['Mechanics'])
        dom_encoded = mlb_dom.fit_transform(data['Domains'])
    else:
        mech_encoded = mlb_mech.transform(data['Mechanics'])
        dom_encoded = mlb_dom.transform(data['Domains'])

    mech_df = pd.DataFrame(mech_encoded, columns=[f"Mech_{m}" for m in mlb_mech.classes_], index=data.index)
    dom_df = pd.DataFrame(dom_encoded, columns=[f"Dom_{d}" for d in mlb_dom.classes_], index=data.index)

    data = pd.concat([data.drop(columns=['Mechanics', 'Domains']), mech_df, dom_df], axis=1)

    return data


def cleaning(data: pd.DataFrame) -> pd.DataFrame:
    data.replace(0, np.nan, inplace=True)
    return data

In [151]:
def fiture_engeniric(data: pd.DataFrame) -> pd.DataFrame:
    # ===== LOGARITHMIC TRANSFORMATIONS =====
    data["Log Year"] = np.log(data["Year Published"] - data["Year Published"].min() + 1)
    data["Log Min Players"] = np.log(data["Min Players"] + 1)
    data["Log Max Players"] = np.log(data["Max Players"] + 1)
    data["Log Play Time"] = np.log(data["Play Time"] + 1)
    data["Log Min Age"] = np.log(data["Min Age"] + 1)
    data["Log Users Rated"] = np.log(data["Users Rated"] + 1)
    data["Log Owned Users"] = np.log(data["Owned Users"] + 1)
    data["Log BGG Rank"] = np.log(data["BGG Rank"] + 1)
    data["Log Complexity Average"] = np.log(data["Complexity Average"] + 0.1)

    # ===== RATIO & NORMALIZATION FEATURES =====
    data["Avg Players"] = (data["Min Players"] + data["Max Players"]) / 2
    data["Player Range"] = data["Max Players"] - data["Min Players"]
    data["Player Flexibility"] = data["Player Range"] / (data["Max Players"] + 1)
    data["Ownership Rate"] = data["Owned Users"] / (data["Users Rated"] + 1)

    # ===== POLYNOMIAL FEATURES =====
    data["Complexity Squared"] = data["Complexity Average"] ** 2
    data["Year Squared"] = (data["Year Published"] - data["Year Published"].min()) ** 2

    # ===== INTERACTION FEATURES =====
    data["Play Time x Min Age"] = data["Play Time"] * data["Min Age"]
    data["Complexity x Players"] = data["Complexity Average"] * data["Avg Players"]

    # ===== CATEGORY FEATURES =====
    max_year = data["Year Published"].max()
    data["Years Since Release"] = max_year - data["Year Published"]
    data["Is Recent"] = (data["Years Since Release"] <= 5).astype(int)
    data["Is Classic"] = (data["Years Since Release"] > 15).astype(int)

    data["Min Age Category"] = pd.cut(data["Min Age"], 
                                      bins=[0, 6, 12, 16, 100], 
                                      labels=[1, 2, 3, 4],
                                      ordered=True).cat.codes

    data["Play Time Category"] = pd.cut(data["Play Time"], 
                                        bins=[0, 30, 60, 120, 10000], 
                                        labels=[1, 2, 3, 4],
                                        ordered=True).cat.codes

    data["Complexity Tier"] = pd.cut(data["Complexity Average"], 
                                     bins=[0, 2, 3, 4, 5], 
                                     labels=[1, 2, 3, 4],
                                     ordered=True).cat.codes

    # ===== POPULARITY & ENGAGEMENT METRICS =====
    data["Popularity Score"] = (data["Log Owned Users"] + data["Log Users Rated"]) / 2
    data["Rank Score"] = 1 / (data["BGG Rank"] / 10000 + 1)

    # ===== STATISTICAL FEATURES =====
    data["Complexity Normalized"] = (data["Complexity Average"] - data["Complexity Average"].min()) / (data["Complexity Average"].max() - data["Complexity Average"].min() + 0.1)
    data["Age Normalized"] = (data["Min Age"] - data["Min Age"].min()) / (data["Min Age"].max() - data["Min Age"].min() + 1)

    # ===== PERCENTILE RANKINGS =====
    data["Popularity Percentile"] = data["Owned Users"].rank(pct=True) * 100
    data["Complexity Percentile"] = data["Complexity Average"].rank(pct=True) * 100

    # ===== DERIVED QUALITY INDICATORS =====
    data["Popular"] = (data["Popularity Percentile"] >= 75).astype(int)
    data["Complex Game"] = (data["Complexity Average"] >= 3).astype(int)
    data["Long Game"] = (data["Play Time"] >= 120).astype(int)
    data["Multiplayer Focused"] = (data["Avg Players"] >= 3).astype(int)

    # ===== COMPOSITE FEATURES =====
    data["Casual Index"] = (1 - data["Complexity Normalized"]) * (1 - data["Age Normalized"] / 100)
    data["Social Index"] = data["Avg Players"] / (data["Max Players"] + 1)

    return data

In [152]:
def pipeline(data: pd.DataFrame, fit: bool=False) -> pd.DataFrame:
    data = convertion(data, fit)
    data = cleaning(data)
    data = fiture_engeniric(data)
    data = expanding(data, fit)
    data = imputing(data, fit)
    
    return data

In [153]:
from sklearn.linear_model import LinearRegression

train = pipeline(train, fit=True)
test = pipeline(test)

linreg = LinearRegression()
linreg.fit(train.drop(columns=["Rating Average"]), train["Rating Average"])
result = linreg.predict(test)

result

array([9.54143084, 9.14260734, 9.5801822 , ..., 3.68752495, 3.65096452,
       3.87724044], shape=(5086,))

In [154]:
pd.Series(result, name="Rating Average").reset_index().to_csv('./games-rating/result.csv', index=False)

In [155]:
!kaggle competitions submit -c games-rating -f ./games-rating/result.csv -m "Message"

100%|█████████████████████████████████████████| 113k/113k [00:00<00:00, 163kB/s]
Successfully submitted to Games Rating